# Build_Gold_Vehicle_Tables

Builds the Gold-layer star-schema tables for vehicle movements — `gold.fact_vehicle_movements` and `gold.dim_route` — from `silver_vehicle_movements`. Feeds the Vehicle Telemetry report page through the shared semantic model.

Runs second in `PL_ColdTrace_Vehicle_Telemetry`, after `Merge_Vehicle_Movements_To_Silver`.

In [4]:
from pyspark.sql import functions as F
from notebookutils import mssparkutils
import json

silver_table_name = "silver_vehicle_movements"
fact_table_name = "gold.fact_vehicle_movements"
route_table_name = "gold.dim_route"
DELAY_THRESHOLD_MINUTES = 5   # a movement counts as delayed past this many minutes late

if not spark.catalog.tableExists(silver_table_name):
    raise ValueError(f"Table '{silver_table_name}' was not found. Run Merge_Vehicle_Movements_To_Silver first.")

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
df_silver = spark.read.table(silver_table_name)

StatementMeta(, 54b541f8-851e-46df-a8e8-290037730eec, 6, Finished, Available, Finished, False)

In [5]:
# Fact table: adds the calendar date (join key for dim_date), the delay in
# minutes (actual event time minus planned arrival), the IsDelayed flag, and
# the IsArrival flag (true when VehicleState signals the vehicle has
# reached a checkpoint). Fully rebuilt from Silver on every run — simple
# and always consistent at this data volume.
df_fact_vehicle_movements = (
    df_silver
    .withColumn("EventDate", F.to_date("EventTimestamp"))
    .withColumn("DelayMinutes",
        (F.unix_timestamp("EventTimestamp") - F.unix_timestamp("PlannedArrival")) / 60.0)
    .withColumn("IsDelayed", F.col("DelayMinutes") > DELAY_THRESHOLD_MINUTES)
    .select("MovementId", "VehicleTripId", "RouteId", "CheckpointId", "EventDate",
            "PlannedArrival", "EventTimestamp", "VehicleState", "DelayMinutes", "IsDelayed")
            .withColumn("IsArrival", F.col("VehicleState") == "Arrived")
)
df_fact_vehicle_movements.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fact_table_name)

StatementMeta(, 54b541f8-851e-46df-a8e8-290037730eec, 7, Finished, Available, Finished, False)

In [6]:
# Dimension table: one row per route, with trip and checkpoint counts seen
# so far — used to slice the fact table by route.
df_dim_route = (
    df_silver.groupBy("RouteId")
    .agg(F.countDistinct("VehicleTripId").alias("TripCount"),
         F.countDistinct("CheckpointId").alias("CheckpointCount"))
)
df_dim_route.write.format("delta").mode("overwrite").saveAsTable(route_table_name)

StatementMeta(, 54b541f8-851e-46df-a8e8-290037730eec, 8, Finished, Available, Finished, False)

In [7]:
# Report status back to the pipeline.
fact_rows = df_fact_vehicle_movements.count()
print(f"Gold tables rebuilt: {fact_table_name} ({fact_rows} rows), {route_table_name}")
mssparkutils.notebook.exit(json.dumps({"status": "gold_build_success", "fact_rows": fact_rows}))

StatementMeta(, 54b541f8-851e-46df-a8e8-290037730eec, 9, Finished, Available, Finished, False)

Gold tables rebuilt: gold.fact_vehicle_movements (16087 rows), gold.dim_route
ExitValue: {"status": "gold_build_success", "fact_rows": 16087}